In [ ]:
import os
import re

import numpy as np
import pandas as pd
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

In [18]:
df = pd.read_csv('../data\\raw/IMDB Dataset.csv')

In [19]:
df.shape

(50000, 2)

In [20]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Lowercase

In [21]:
df['review'] = df['review'].str.lower()
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


## Remove html

In [22]:
import re
def remove_html(text):
    pattern =  re.compile('<.*?>')
    return pattern.sub(r'', text)
df['review'] = df['review'].apply(remove_html)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Remove url

In [23]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)
df['review'] = df['review'].apply(remove_url)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Remove Mention

In [24]:
def remove_mention(text):
    pattern = re.compile(r'(?<!\w)@\w+')
    return pattern.sub(r'', text)
df['review'] = df['review'].apply(remove_mention)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Remove hashtag

In [25]:
def remove_hashtag(text):
    pattern = re.compile(r'(?<!\w)#\w+')
    return pattern.sub(r'', text)
df['review'] = df['review'].apply(remove_hashtag)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Remove special characters

In [26]:
def remove_special_characters(text, keep_apostrophe=True):
    # Keep letters/spaces; optionally keep apostrophes in words like don't.
    if keep_apostrophe:
        pattern = re.compile(r"[^\w\s']")
    else:
        pattern = re.compile(r'[^\w\s]')
    return pattern.sub('', text)

df['review'] = df['review'].apply(remove_special_characters)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production the filming tech...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's love in the time of money is a...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Remove numbers (conditional)

In [27]:
def remove_numbers(text, mode='standalone'):
    # mode='standalone': remove tokens that are only digits (e.g., 2024, 10/10 -> 10 parts).
    # mode='all': remove every digit character, including inside words (movie2 -> movie).
    if mode == 'all':
        pattern = re.compile(r'\d+')
    else:
        pattern = re.compile(r'\b\d+\b')
    return pattern.sub('', text)

# Change mode to 'all' if you want to remove every digit character.
df['review'] = df['review'].apply(lambda x: remove_numbers(x, mode='standalone'))
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production the filming tech...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's love in the time of money is a...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## Normalize whitespace

In [28]:
def normalize_whitespace(text):
    # Replace repeated spaces/newlines/tabs with a single space, then trim.
    return re.sub(r'\s+', ' ', text).strip()

df['review'] = df['review'].apply(normalize_whitespace)
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production the filming tech...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's love in the time of money is a...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: str

## a) Pipeline chuẩn hóa văn bản

### Mục tiêu
- Chuẩn hóa văn bản theo chuỗi bước nhất quán: lowercase, xóa HTML/URL/mention/hashtag, xóa ký tự đặc biệt, xóa số có điều kiện, chuẩn hóa khoảng trắng.
- Định lượng tác động của từng bước lên không gian từ vựng và độ dài văn bản.

### Thiết kế báo cáo
- Với mỗi bước tiền xử lý, ghi nhận:
  1. `prev_vocab_size`, `curr_vocab_size`, `vocab_change_ratio`.
  2. Thống kê độ dài văn bản theo token và ký tự: mean, median, p90.
- Dữ liệu được chạy tuần tự từ trạng thái `raw` để bảo toàn tính nhân quả của từng bước.

In [29]:
# Rebuild from raw text to evaluate each preprocessing step quantitatively.
df_eval = pd.read_csv('../data\\raw/IMDB Dataset.csv').copy()
df_eval['review_raw'] = df_eval['review'].astype(str)

def tokenize(series):
    return series.str.split()

def vocab_set(series):
    tokens = tokenize(series).explode().dropna()
    return set(tokens.tolist())

def length_stats(series):
    word_len = tokenize(series).str.len()
    char_len = series.str.len()
    return {
        'doc_count': int(series.shape[0]),
        'word_len_mean': float(word_len.mean()),
        'word_len_median': float(word_len.median()),
        'word_len_p90': float(word_len.quantile(0.9)),
        'char_len_mean': float(char_len.mean()),
        'char_len_median': float(char_len.median()),
        'char_len_p90': float(char_len.quantile(0.9)),
    }

def vocab_change_ratio(prev_series, curr_series):
    prev_vocab = vocab_set(prev_series)
    curr_vocab = vocab_set(curr_series)
    if len(prev_vocab) == 0:
        return {'prev_vocab_size': 0, 'curr_vocab_size': len(curr_vocab), 'vocab_change_ratio': 0.0}
    overlap = len(prev_vocab & curr_vocab)
    changed_ratio = 1 - (overlap / len(prev_vocab))
    return {
        'prev_vocab_size': len(prev_vocab),
        'curr_vocab_size': len(curr_vocab),
        'vocab_change_ratio': float(changed_ratio),
    }

steps = [
    ('lowercase', lambda s: s.str.lower()),
    ('remove_html', lambda s: s.apply(remove_html)),
    ('remove_url', lambda s: s.apply(remove_url)),
    ('remove_mention', lambda s: s.apply(remove_mention)),
    ('remove_hashtag', lambda s: s.apply(remove_hashtag)),
    ('remove_special_characters', lambda s: s.apply(lambda x: remove_special_characters(x, keep_apostrophe=True))),
    ('remove_numbers_standalone', lambda s: s.apply(lambda x: remove_numbers(x, mode='standalone'))),
    ('normalize_whitespace', lambda s: s.apply(normalize_whitespace)),
]

report_rows = []
prev_name = 'raw'
prev_series = df_eval['review_raw'].copy()
report_rows.append({
    'step': prev_name,
    'prev_vocab_size': np.nan,
    'curr_vocab_size': len(vocab_set(prev_series)),
    'vocab_change_ratio': np.nan,
    **length_stats(prev_series),
})

for step_name, step_func in steps:
    curr_series = step_func(prev_series)
    vocab_metrics = vocab_change_ratio(prev_series, curr_series)
    stats = length_stats(curr_series)
    report_rows.append({
        'step': step_name,
        **vocab_metrics,
        **stats,
    })
    prev_series = curr_series

report_df = pd.DataFrame(report_rows)
report_df['vocab_change_ratio'] = report_df['vocab_change_ratio'].round(4)
for col in ['word_len_mean', 'word_len_median', 'word_len_p90', 'char_len_mean', 'char_len_median', 'char_len_p90']:
    report_df[col] = report_df[col].round(2)

report_df

,step,prev_vocab_size,curr_vocab_size,vocab_change_ratio,doc_count,word_len_mean,word_len_median,word_len_p90,char_len_mean,char_len_median,char_len_p90
0,raw,NaN,438729,NaN,50000,231.16,173.0,451.0,1309.43,970.0,2581.0
1,lowercase,438729.0,390931,0.4418,50000,231.16,173.0,451.0,1309.43,970.0,2581.0
2,remove_html,390931.0,422479,0.0886,50000,227.11,170.0,443.0,1285.17,953.0,2528.0
3,remove_url,422479.0,422303,0.0004,50000,227.11,170.0,443.0,1285.03,953.0,2528.0
4,remove_mention,422303.0,422301,0.0000,50000,227.11,170.0,443.0,1285.03,953.0,2528.0
5,remove_hashtag,422301.0,422185,0.0004,50000,227.10,170.0,443.0,1285.01,953.0,2528.0
6,remove_special_characters,422185.0,236072,0.7519,50000,226.19,170.0,441.0,1249.79,927.0,2458.1
7,remove_numbers_standalone,236072.0,234035,0.0093,50000,225.20,169.0,439.0,1247.22,925.0,2454.1
8,normalize_whitespace,234035.0,234035,0.0000,50000,225.20,169.0,439.0,1245.28,923.0,2451.0


### Phân tích tác động lên phân phối độ dài theo từng bước

- Bảng dưới đây bổ sung các cột chênh lệch (`delta`) giữa hai bước liên tiếp.
- Mục tiêu là xác định bước nào làm thay đổi mạnh nhất phần thân phân phối (mean) và phần đuôi phân phối (p90).

In [30]:
impact_df = report_df.copy()
impact_df['word_len_mean_delta'] = impact_df['word_len_mean'].diff().round(2)
impact_df['word_len_p90_delta'] = impact_df['word_len_p90'].diff().round(2)
impact_df['char_len_mean_delta'] = impact_df['char_len_mean'].diff().round(2)
impact_df['char_len_p90_delta'] = impact_df['char_len_p90'].diff().round(2)

impact_df[[
    'step',
    'vocab_change_ratio',
    'word_len_mean', 'word_len_mean_delta',
    'word_len_p90', 'word_len_p90_delta',
    'char_len_mean', 'char_len_mean_delta',
    'char_len_p90', 'char_len_p90_delta'
]]

,step,vocab_change_ratio,word_len_mean,word_len_mean_delta,word_len_p90,word_len_p90_delta,char_len_mean,char_len_mean_delta,char_len_p90,char_len_p90_delta
0,raw,NaN,231.16,NaN,451.0,NaN,1309.43,NaN,2581.0,NaN
1,lowercase,0.4418,231.16,0.00,451.0,0.0,1309.43,0.00,2581.0,0.0
2,remove_html,0.0886,227.11,-4.05,443.0,-8.0,1285.17,-24.26,2528.0,-53.0
3,remove_url,0.0004,227.11,0.00,443.0,0.0,1285.03,-0.14,2528.0,0.0
4,remove_mention,0.0000,227.11,0.00,443.0,0.0,1285.03,0.00,2528.0,0.0
5,remove_hashtag,0.0004,227.10,-0.01,443.0,0.0,1285.01,-0.02,2528.0,0.0
6,remove_special_characters,0.7519,226.19,-0.91,441.0,-2.0,1249.79,-35.22,2458.1,-69.9
7,remove_numbers_standalone,0.0093,225.20,-0.99,439.0,-2.0,1247.22,-2.57,2454.1,-4.0
8,normalize_whitespace,0.0000,225.20,0.00,439.0,0.0,1245.28,-1.94,2451.0,-3.1


## b) So sánh chiến lược tokenization

### Mục tiêu
- So sánh 4 nhóm chiến lược: word-level, sentence-level, character-level, và subword (BPE).
- Với word-level, thực nghiệm cả hai tokenizer: NLTK và spaCy.

### Chỉ số đánh giá
- `vocab_size`: kích thước từ vựng trên tập train.
- `oov_ratio_test`: tỉ lệ token ở test không xuất hiện trong từ vựng train.
- `avg_token_seq_len_test`: độ dài chuỗi token trung bình trên tập test.

### Cài đặt
- Dữ liệu đầu vào được làm sạch nhẹ để đảm bảo công bằng giữa các tokenizer.
- BPE huấn luyện bằng thư viện `tokenizers` của Hugging Face.

In [35]:
import re
import numpy as np
import pandas as pd
import nltk
import spacy

from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize, sent_tokenize
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Lightweight spaCy tokenizer-only pipeline.
nlp_spacy = spacy.blank('en')

df_tok = pd.read_csv('../data\\raw/IMDB Dataset.csv').copy()

def basic_clean_for_tokenization(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?<!\w)@\w+', ' ', text)
    text = re.sub(r'(?<!\w)#\w+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_tok['text_clean'] = df_tok['review'].apply(basic_clean_for_tokenization)

train_texts, test_texts = train_test_split(
    df_tok['text_clean'],
    test_size=0.2,
    random_state=42,
    shuffle=True,
 )

train_text_list = train_texts.tolist()
test_text_list = test_texts.tolist()

def flatten_token_lists(token_lists):
    return [tok for doc in token_lists for tok in doc]

def oov_ratio(test_token_lists, train_vocab):
    test_tokens = flatten_token_lists(test_token_lists)
    if len(test_tokens) == 0:
        return 0.0
    oov_count = sum(tok not in train_vocab for tok in test_tokens)
    return oov_count / len(test_tokens)

def evaluate_strategy(name, train_token_lists, test_token_lists):
    train_vocab = set(flatten_token_lists(train_token_lists))
    avg_len_test = float(np.mean([len(x) for x in test_token_lists]))
    return {
        'strategy': name,
        'vocab_size': int(len(train_vocab)),
        'oov_ratio_test': float(oov_ratio(test_token_lists, train_vocab)),
        'avg_token_seq_len_test': avg_len_test,
    }

def spacy_word_tokenize(text):
    return [tok.text for tok in nlp_spacy.make_doc(text)]

# Four strategy families (with two implementations for word-level).
strategy_tokenizers = {
    'word_level_nltk': word_tokenize,
    'word_level_spacy': spacy_word_tokenize,
    'sentence_level': sent_tokenize,
    'character_level': list,
}

comparison_rows = []
for strategy_name, tokenizer_fn in strategy_tokenizers.items():
    train_tokens = [tokenizer_fn(text) for text in train_text_list]
    test_tokens = [tokenizer_fn(text) for text in test_text_list]
    comparison_rows.append(evaluate_strategy(strategy_name, train_tokens, test_tokens))

# Subword (BPE using Hugging Face tokenizers).
bpe_tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
bpe_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
)
bpe_tokenizer.train_from_iterator(train_text_list, trainer=bpe_trainer)

train_bpe = [bpe_tokenizer.encode(text).tokens for text in train_text_list]
test_bpe = [bpe_tokenizer.encode(text).tokens for text in test_text_list]
comparison_rows.append(evaluate_strategy('subword_bpe_hf_tokenizers', train_bpe, test_bpe))

tokenization_comparison_df = pd.DataFrame(comparison_rows)
tokenization_comparison_df['oov_ratio_test'] = tokenization_comparison_df['oov_ratio_test'].round(6)
tokenization_comparison_df['avg_token_seq_len_test'] = tokenization_comparison_df['avg_token_seq_len_test'].round(2)

tokenization_comparison_df

,strategy,vocab_size,oov_ratio_test,avg_token_seq_len_test
0,word_level_nltk,134558,0.007295,265.54
1,word_level_spacy,110749,0.005472,269.26
2,sentence_level,470742,0.955044,12.14
3,character_level,158,0.000000,1291.00
4,subword_bpe_hf_tokenizers,15883,0.000008,288.35


In [36]:
# Quick dependency check in current notebook kernel
deps = ['numpy', 'pandas', 'nltk', 'spacy', 'sklearn', 'tokenizers']
missing = []
versions = {}

for pkg in deps:
    try:
        mod = __import__(pkg)
        versions[pkg] = getattr(mod, '__version__', 'unknown')
    except Exception as e:
        missing.append((pkg, str(e)))

print('VERSIONS:', versions)
print('MISSING_OR_BROKEN:', missing)

VERSIONS: {'numpy': '2.4.4', 'pandas': '3.0.2', 'nltk': '3.9.4', 'spacy': '3.8.14', 'sklearn': '1.8.0', 'tokenizers': '0.22.2'}
MISSING_OR_BROKEN: []


## c) Loại bỏ stop words và phân tích thông tin

### Mục tiêu thực nghiệm
- So sánh ảnh hưởng của việc loại bỏ stop words lên không gian đặc trưng và hiệu năng mô hình phân loại cảm xúc.
- Báo cáo đầy đủ 3 tiêu chí theo yêu cầu:
  1. Kích thước từ vựng trước/sau loại bỏ stop words.
  2. Mutual Information (MI) trung bình giữa từ và nhãn trước/sau.
  3. Hiệu năng Naive Bayes trước/sau.

### Thiết kế thực nghiệm
- Dữ liệu được chuẩn hóa nhẹ (lowercase, bỏ HTML/URL/mention/hashtag, chuẩn hóa khoảng trắng) để đảm bảo công bằng khi so sánh.
- Chia train/test theo tỉ lệ 80/20, có `stratify` theo nhãn.
- Dùng `CountVectorizer` và `MultinomialNB` cho cả hai kịch bản:
  - `before_stopword_removal`: giữ toàn bộ từ.
  - `after_stopword_removal`: loại bỏ stop words tiếng Anh của NLTK.
- MI được tính bằng `mutual_info_classif` trên đặc trưng BoW (discrete features).

In [37]:
import re
import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

nltk.download('stopwords', quiet=True)

def clean_text_for_stopword_study(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?<!\w)@\w+', ' ', text)
    text = re.sub(r'(?<!\w)#\w+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_stop = pd.read_csv('../data\\raw/IMDB Dataset.csv').copy()
df_stop['text_clean'] = df_stop['review'].apply(clean_text_for_stopword_study)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_stop['text_clean'],
    df_stop['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df_stop['sentiment']
 )

stop_words_en = set(stopwords.words('english'))
y_train_bin = (y_train == 'positive').astype(int).to_numpy()

def evaluate_stopword_setting(remove_stopwords):
    vectorizer = CountVectorizer(
        stop_words=sorted(stop_words_en) if remove_stopwords else None,
        token_pattern=r'(?u)\b\w+\b',
        min_df=2
    )

    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    vocab_size = len(vectorizer.vocabulary_)

    mi_scores = mutual_info_classif(
        X_train,
        y_train_bin,
        discrete_features=True,
        random_state=42
    )
    mean_mi = float(np.mean(mi_scores))

    clf = MultinomialNB()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    return {
        'vocab_size': int(vocab_size),
        'mean_mi': float(mean_mi),
        'nb_accuracy': float(accuracy_score(y_test, y_pred)),
        'nb_f1_macro': float(f1_score(y_test, y_pred, average='macro'))
    }

before_stats = evaluate_stopword_setting(remove_stopwords=False)
after_stats = evaluate_stopword_setting(remove_stopwords=True)

stopword_comparison_df = pd.DataFrame([
    {
        'metric': 'vocab_size',
        'before': before_stats['vocab_size'],
        'after': after_stats['vocab_size'],
        'delta_after_minus_before': after_stats['vocab_size'] - before_stats['vocab_size']
    },
    {
        'metric': 'mean_mi',
        'before': before_stats['mean_mi'],
        'after': after_stats['mean_mi'],
        'delta_after_minus_before': after_stats['mean_mi'] - before_stats['mean_mi']
    },
    {
        'metric': 'nb_accuracy',
        'before': before_stats['nb_accuracy'],
        'after': after_stats['nb_accuracy'],
        'delta_after_minus_before': after_stats['nb_accuracy'] - before_stats['nb_accuracy']
    },
    {
        'metric': 'nb_f1_macro',
        'before': before_stats['nb_f1_macro'],
        'after': after_stats['nb_f1_macro'],
        'delta_after_minus_before': after_stats['nb_f1_macro'] - before_stats['nb_f1_macro']
    }
])

stopword_comparison_df

,metric,before,after,delta_after_minus_before
0,vocab_size,54777.000000,54625.000000,-152.000000
1,mean_mi,0.000088,0.000083,-0.000004
2,nb_accuracy,0.844100,0.862500,0.018400
3,nb_f1_macro,0.843953,0.862450,0.018497


In [38]:
acc_delta = after_stats['nb_accuracy'] - before_stats['nb_accuracy']
f1_delta = after_stats['nb_f1_macro'] - before_stats['nb_f1_macro']
mi_delta = after_stats['mean_mi'] - before_stats['mean_mi']

print('_' * 80)
print('THAO LUAN: Stop words co luon cai thien ket qua khong?')
print('_' * 80)
print(f"- Vocab size thay doi: {before_stats['vocab_size']:,} -> {after_stats['vocab_size']:,}")
print(f"- Mean MI thay doi  : {before_stats['mean_mi']:.6f} -> {after_stats['mean_mi']:.6f} (delta={mi_delta:+.6f})")
print(f"- NB Accuracy       : {before_stats['nb_accuracy']:.4f} -> {after_stats['nb_accuracy']:.4f} (delta={acc_delta:+.4f})")
print(f"- NB F1-macro       : {before_stats['nb_f1_macro']:.4f} -> {after_stats['nb_f1_macro']:.4f} (delta={f1_delta:+.4f})")

if acc_delta > 0 and f1_delta > 0:
    print('\nNhan xet: Loai bo stop words giup mo hinh tot hon trong thuc nghiem nay.')
elif acc_delta < 0 and f1_delta < 0:
    print('\nNhan xet: Loai bo stop words lam giam hieu nang; stop words KHONG luon co hai.')
else:
    print('\nNhan xet: Tac dong la hon hop; stop words KHONG luon cai thien ket qua.')

print('\nLy do tong quat:')
print('- Cac tu phu dinh nhu not/no/nor co the mang thong tin cam xuc manh; neu bi xoa co the gay mat ngu nghia.')
print('- Trong bai toan sentiment, can can nhac giu lai nhom tu phu dinh hoac danh sach stop words tuy chinh.')
print('_' * 80)

________________________________________________________________________________
THAO LUAN: Stop words co luon cai thien ket qua khong?
________________________________________________________________________________
- Vocab size thay doi: 54,777 -> 54,625
- Mean MI thay doi  : 0.000088 -> 0.000083 (delta=-0.000004)
- NB Accuracy       : 0.8441 -> 0.8625 (delta=+0.0184)
- NB F1-macro       : 0.8440 -> 0.8624 (delta=+0.0185)

Nhan xet: Loai bo stop words giup mo hinh tot hon trong thuc nghiem nay.

Ly do tong quat:
- Cac tu phu dinh nhu not/no/nor co the mang thong tin cam xuc manh; neu bi xoa co the gay mat ngu nghia.
- Trong bai toan sentiment, can can nhac giu lai nhom tu phu dinh hoac danh sach stop words tuy chinh.
________________________________________________________________________________
